In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
# 假设你的项目使用了 logging
import logging
logger = logging.getLogger(__name__)
# 如果没有配置 logging，可以注释掉 logger 相关行，或者添加基本配置：
# logging.basicConfig(level=logging.INFO)

# 假设 cfg 是一个类似字典的对象，例如 OmegaConf 的 DictConfig
# from omegaconf import DictConfig # 如果你使用 OmegaConf

def plot_spatial_rmse_optimized(
    spatial_rmse: np.ndarray,
    cfg: dict, # 使用普通字典作为示例，你可以传入 DictConfig
    mask: np.ndarray,
    lon: np.ndarray, # 经度数据现在作为参数传入
    lat: np.ndarray, # 纬度数据现在作为参数传入
    save_path: str,
    # --- 新增或修改的参数 ---
    title: str = "空间RMSE分布 (Spatial RMSE Distribution)",
    cmap_name: str = "magma", # 默认使用 'magma' (感知均匀)
    colorbar_label: str = "RMSE (PSU)", # 包含单位
    colorbar_extend: str = 'neither', # 'neither', 'min', 'max', 'both'
    colorbar_nticks: int = 5, # 颜色条刻度数量
    gridline_fontsize: int = 10, # 网格标签字体大小
    title_fontsize: int = 14, # 标题字体大小
    shading: str = 'auto' # 'auto', 'flat', 'gouraud'
):
    """
    使用 Cartopy 绘制优化后的空间 RMSE 分布图。

    Args:
        spatial_rmse (np.ndarray): 包含空间RMSE值的二维数组。
        cfg (dict): 配置字典，包含 visualization.map, evaluation.spatial 等子项。
        mask (np.ndarray): 布尔数组，True表示有效水域，False表示陆地/无效区域。
        lon (np.ndarray): 经度坐标数组 (一维或二维)。
        lat (np.ndarray): 纬度坐标数组 (一维或二维)。
        save_path (str): 图像保存路径。
        title (str): 图表标题 (建议简洁)。
        cmap_name (str): Matplotlib 颜色映射方案名称。
        colorbar_label (str): 颜色条标签文字。
        colorbar_extend (str): 颜色条末端箭头样式。
        colorbar_nticks (int): 颜色条期望的刻度数量。
        gridline_fontsize (int): 经纬度网格标签字体大小。
        title_fontsize (int): 图表标题字体大小。
        shading (str): pcolormesh 的 shading 参数。
    """
    map_cfg = cfg.get('visualization', {}).get('map', {})
    eval_cfg = cfg.get('evaluation', {}).get('spatial', {})
    vis_cfg = cfg.get('visualization', {})

    # --- 1. 获取地图投影和数据 CRS ---
    # 假设: 数据是标准的经纬度网格 (PlateCarree)
    # 投影方式可以通过 cfg 指定，例如 'Mercator' 或 'PlateCarree'
    proj_name = map_cfg.get('projection', 'PlateCarree')
    if proj_name == 'Mercator':
        map_proj = ccrs.Mercator()
    else: # 默认为 PlateCarree
        map_proj = ccrs.PlateCarree()
        if proj_name != 'PlateCarree':
            logger.warning(f"未知的投影 '{proj_name}'，将使用 PlateCarree。")

    # 假设: 输入的 lon, lat 数据是标准的地理坐标 (WGS84)
    data_crs = ccrs.PlateCarree()

    # --- 2. 创建 Figure 和 GeoAxes ---
    fig = plt.figure(figsize=tuple(map_cfg.get('figsize', (10, 8))))
    ax = fig.add_subplot(1, 1, 1, projection=map_proj)

    # --- 3. 设置地图范围 ---
    extent = map_cfg.get('extent') # 例如: [112.5, 114.5, 21.5, 23.0]
    if extent:
        try:
            # Cartopy 期望 extent 的坐标系与地图投影的坐标系一致
            # 但 set_extent 有 crs 参数，可以指定 extent 值的坐标系
            ax.set_extent(extent, crs=data_crs)
        except Exception as e:
            logger.error(f"设置地图范围失败: {e}. extent={extent}, data_crs={data_crs}")
            plt.close(fig)
            return
    else:
        logger.warning("未在配置中找到地图范围 'extent'，将自动确定范围。")


    # --- 4. 添加地图特征 (直接实现) ---
    try:
        # 添加陆地，设置颜色
        land_color = map_cfg.get("land_color", "lightgrey") # 可配置陆地颜色
        ax.add_feature(cfeature.LAND, facecolor=land_color, zorder=1)
        # 添加海岸线
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=2)
        # 可选：添加河流 (如果需要且 cartopy 能获取到)
        if map_cfg.get("add_rivers", False):
            ax.add_feature(cfeature.RIVERS, zorder=2)
        # 可选：添加边界 (如果需要)
        if map_cfg.get("add_borders", False):
             ax.add_feature(cfeature.BORDERS, linestyle=':', zorder=2)
    except Exception as e:
        logger.error(f"添加地图要素时出错: {e}")
        # 可以选择继续绘制或返回

    # --- 5. 准备要绘制的数据和颜色映射 ---
    # 使用传入的 mask，True代表有效区域 (水域)，False代表无效 (陆地)
    # np.ma.masked_where 的第一个参数是 mask 条件，True 表示要 mask 掉的地方
    data_to_plot = np.ma.masked_where(~mask, spatial_rmse)

    # 获取配置的或默认的 colormap
    cmap_name = eval_cfg.get("colormap", cmap_name) # 优先使用函数参数，其次cfg，最后默认
    try:
        cmap = plt.get_cmap(cmap_name)
        # 设置无效区域颜色 (对应 ~mask 为 True 的地方)
        cmap.set_bad(color=map_cfg.get("mask_color", 'darkgray')) # 使用与陆地不同的颜色更好区分
    except ValueError:
        logger.warning(f"颜色映射方案 '{cmap_name}' 不存在，将使用 'viridis'。")
        cmap_name = 'viridis'
        cmap = plt.get_cmap(cmap_name)
        cmap.set_bad(color=map_cfg.get("mask_color", 'darkgray'))


    # 确定颜色映射范围 vmin, vmax
    vmin = eval_cfg.get("vmin")
    vmax = eval_cfg.get("vmax")
    if vmin is None or vmax is None:
        valid_data = data_to_plot[~data_to_plot.mask]
        if valid_data.size > 0:
            # 自动计算范围，可以考虑使用百分位数排除极端值，例如 5% 和 95%
            auto_vmin = np.percentile(valid_data, eval_cfg.get("vmin_percentile", 1))
            auto_vmax = np.percentile(valid_data, eval_cfg.get("vmax_percentile", 99))
            if vmin is None: vmin = auto_vmin
            if vmax is None: vmax = auto_vmax
            logger.info(f"自动计算得到 vmin={vmin:.2f} (基于{eval_cfg.get('vmin_percentile', 1)}%) 和 vmax={vmax:.2f} (基于{eval_cfg.get('vmax_percentile', 99)}%)")
        else:
            vmin, vmax = 0, 1 # Fallback if no valid data
            logger.warning("无法自动计算 vmin/vmax，因为没有有效的 RMSE 数据。将使用默认范围 [0, 1]。")
    # 确保 vmin < vmax
    if vmin >= vmax:
        logger.warning(f"vmin ({vmin}) >= vmax ({vmax})，将使用默认范围 [0, 1] 或调整。")
        vmin, vmax = 0, 1 # 或者根据数据调整

    # --- 6. 绘制数据 (pcolormesh) ---
    try:
        # 使用传入的 shading 参数
        shading_option = eval_cfg.get("shading", shading) # 优先函数参数，其次cfg，最后默认
        pcm = ax.pcolormesh(lon, lat, data_to_plot,
                            transform=data_crs, # 关键：告知 Cartopy 数据的坐标系
                            cmap=cmap,
                            vmin=vmin,
                            vmax=vmax,
                            shading=shading_option,
                            zorder=3) # 让数据绘制在要素之上
        logger.debug(f"绘制 pcolormesh: vmin={vmin}, vmax={vmax}, cmap={cmap_name}, shading={shading_option}")
    except Exception as e:
        logger.error(f"绘制 pcolormesh 失败: {e}")
        plt.close(fig)
        return

    # --- 7. 添加颜色条 (直接实现) ---
    try:
        cb = fig.colorbar(pcm, ax=ax, orientation='vertical', fraction=0.04, pad=0.04,
                          extend=colorbar_extend, label=colorbar_label)
        cb.ax.tick_params(labelsize=gridline_fontsize) # 设置颜色条刻度字体
        cb.set_label(colorbar_label, size=gridline_fontsize + 1) # 设置颜色条标签字体

        # 控制颜色条刻度数量
        if colorbar_nticks is not None:
            tick_locator = mticker.MaxNLocator(nbins=colorbar_nticks)
            cb.locator = tick_locator
            cb.update_ticks()
    except Exception as e:
        logger.error(f"添加颜色条失败: {e}")

    # --- 8. 添加网格线 (直接实现) ---
    try:
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False # 不显示顶部标签
        gl.right_labels = False # 不显示右侧标签
        # 控制标签格式和字体大小
        gl.xformatter = mticker.FormatStrFormatter('%.2f°E') # 格式化经度标签
        gl.yformatter = mticker.FormatStrFormatter('%.2f°N') # 格式化纬度标签
        gl.xlabel_style = {'size': gridline_fontsize, 'color': 'black'}
        gl.ylabel_style = {'size': gridline_fontsize, 'color': 'black'}
    except Exception as e:
        logger.error(f"添加网格线失败: {e}")

    # --- 9. 设置标题和保存 ---
    ax.set_title(title, fontsize=title_fontsize)

    # --- 10. 调整布局和保存 ---
    try:
        plt.tight_layout()
    except Exception as e:
        logger.warning(f"执行 tight_layout 时出错: {e}")

    if save_path:
        try:
            # 确保保存路径的目录存在
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            plt.savefig(save_path, dpi=vis_cfg.get("dpi", 300), bbox_inches='tight')
            logger.info(f"空间 RMSE 图已保存到: {save_path}")
        except Exception as e:
            logger.error(f"保存图像失败 {save_path}: {e}")

    # 根据配置决定是否显示图形 (适用于 notebook 等环境)
    if vis_cfg.get("show_figures", False):
        plt.show()

    plt.close(fig) # [代码用中文注释] 无论是否显示，最后都关闭图形，释放内存

# --- 示例用法 (需要你有相应的数据和配置) ---
if __name__ == '__main__':
    # 假设你已经加载了以下数据：
    # spatial_rmse_data: (height, width) 的 numpy 数组
    # mask_data: (height, width) 的布尔数组 (True 为水域)
    # lon_data: (height, width) 或 (width,) 的经度数组
    # lat_data: (height, width) 或 (height,) 的纬度数组
    # my_cfg: 包含配置的字典

    # 示例: 创建虚拟数据和配置
    logger.info("开始绘制示例空间 RMSE 图...")
    height, width = 100, 80
    lon_data = np.linspace(112.5, 114.5, width)
    lat_data = np.linspace(21.5, 23.0, height)
    lon_grid, lat_grid = np.meshgrid(lon_data, lat_data)

    # 创建一个简单的圆形水域mask
    center_x, center_y = width // 2, height // 2
    radius = min(center_x, center_y) - 10
    y_coords, x_coords = np.ogrid[:height, :width]
    mask_data = (x_coords - center_x)**2 + (y_coords - center_y)**2 <= radius**2

    # 创建模拟RMSE数据，中心高，向外递减
    dist_from_center = np.sqrt((x_coords - center_x)**2 + (y_coords - center_y)**2)
    spatial_rmse_data = 5 * np.exp(-dist_from_center / (radius / 2)) + np.random.rand(height, width) * 0.5
    spatial_rmse_data[~mask_data] = 0 # 陆地设为0 (会被mask掉)

    # 示例配置
    my_cfg = {
        'visualization': {
            'map': {
                'figsize': (8, 7),
                'extent': [112.5, 114.5, 21.5, 23.0], # 地图范围
                'projection': 'PlateCarree',
                'land_color': '#E0E0E0', # 浅灰色陆地
                'mask_color': 'white',   # 掩码区域白色
                'add_rivers': False,
                'add_borders': False
            },
            'dpi': 150,
            'show_figures': False # 不在运行时显示，直接保存
        },
        'evaluation': {
            'spatial': {
                'colormap': 'plasma', # 使用 plasma 色图
                # 'vmin': 0,       # 可以手动指定 vmin
                # 'vmax': 6,       # 可以手动指定 vmax
                'vmin_percentile': 0, # 使用最小值作为vmin
                'vmax_percentile': 100,# 使用最大值作为vmax
                'shading': 'gouraud' # 使用平滑着色
            }
        }
    }

    # 调用优化后的函数
    plot_spatial_rmse_optimized(
        spatial_rmse=spatial_rmse_data,
        cfg=my_cfg,
        mask=mask_data,
        lon=lon_grid, # 注意传入网格化的经纬度
        lat=lat_grid,
        save_path="spatial_rmse_optimized.png",
        title="空间RMSE分布 (模拟数据)", # 简洁标题
        cmap_name="plasma", # 覆盖配置中的 colormap
        colorbar_label="模拟RMSE (单位)",
        colorbar_extend='neither',
        colorbar_nticks=6,
        gridline_fontsize=9,
        title_fontsize=12,
        shading='gouraud' # 覆盖配置中的 shading
    )
    logger.info("示例图绘制完成。")

/home/lirz6/miniconda3/envs/presal/lib/python3.12/site-packages/numpy/lib/function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
/home/lirz6/miniconda3/envs/presal/lib/python3.12/site-packages/cartopy/mpl/geoaxes.py:527: UserWarning: Glyph 31354 (\N{CJK UNIFIED IDEOGRAPH-7A7A}) missing from current font.
  super()._update_title_position(renderer)
/home/lirz6/miniconda3/envs/presal/lib/python3.12/site-packages/cartopy/mpl/geoaxes.py:527: UserWarning: Glyph 38388 (\N{CJK UNIFIED IDEOGRAPH-95F4}) missing from current font.
  super()._update_title_position(renderer)
/home/lirz6/miniconda3/envs/presal/lib/python3.12/site-packages/cartopy/mpl/geoaxes.py:527: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from current font.
  super()._update_title_position(renderer)
/home/lirz6/miniconda3/envs/presal/lib/python3.12/site-packages/cartopy/mpl/geoaxes.py:527: UserWarning: Glyph 24067 (\N{CJK UNIFIED IDEOGRAPH-5E03})